In [ ]:
# generates the plots of replication time and inter-origin distance, as stratified by a signature

In [29]:
import pickle
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import seaborn as sns

In [30]:
# load the NNMF

In [31]:

exp_name='non_clustered'
dataset = '_pancan_sel'
pkl_file = '/home/dg204/park_dglodzik/svig/extractionLogs/SV_SigExtraction_nmf_RFD_'+exp_name+dataset+'.pkl'
X_file = '/home/dg204/park_dglodzik/svig/extractionLogs/X_'+exp_name+dataset+'.csv'
with open(pkl_file, 'rb') as f:
    model_loaded = pickle.load(f)
    deNovoSigs = pd.DataFrame(model_loaded.W)
    H_sel = pd.DataFrame(model_loaded.H.T)
    sil_scores = model_loaded.sil_score
X = pd.read_csv(X_file, index_col=0)

deNovoSigs.columns = [f"Signature {i+1}" for i in range(deNovoSigs.shape[1])]
H_sel.columns = [f"Signature {i+1}" for i in range(deNovoSigs.shape[1])]
H_sel.index = model_loaded.samples
deNovoSigs.index = model_loaded.features
deNovoSigs_expanded = deNovoSigs.reindex(X.index)
deNovoSigs = deNovoSigs_expanded

In [32]:
sample_channel_sig_assignments = []
for sam in model_loaded.samples:
    sam_exp = H_sel.loc[sam,:]
    sam_exp = sam_exp/sam_exp.sum()
    sam_exp_m = pd.DataFrame(np.tile(sam_exp.to_numpy(), (deNovoSigs.shape[0], 1)))
    sam_exp_m.index =     deNovoSigs.index
    sam_exp_m.columns=deNovoSigs.columns
    
    psig_by_channel = deNovoSigs * sam_exp_m
    highest_sig_per_channel = pd.DataFrame(psig_by_channel.idxmax(axis=1))
    highest_sig_per_channel = highest_sig_per_channel.reset_index()
    highest_sig_per_channel['sample'] = sam
    sample_channel_sig_assignments.append(highest_sig_per_channel)

In [33]:
all_sample_channel_assignments = pd.concat(sample_channel_sig_assignments, ignore_index=True)
all_sample_channel_assignments.columns = ['label_RFD', 'sigID', 'sample']

In [34]:
all_sample_channel_assignments.head()

,label_RFD,sigID,sample
0,non-clustered_del_1-10Kb_L_L,Signature 10,8f558713-f32b-403b-aedf-c79efeb41c67
1,non-clustered_del_1-10Kb_L_R,Signature 11,8f558713-f32b-403b-aedf-c79efeb41c67
2,non-clustered_del_1-10Kb_R_L,Signature 10,8f558713-f32b-403b-aedf-c79efeb41c67
3,non-clustered_del_1-10Kb_R_R,Signature 10,8f558713-f32b-403b-aedf-c79efeb41c67
4,non-clustered_del_10-100Kb_L_L,Signature 11,8f558713-f32b-403b-aedf-c79efeb41c67


In [35]:
# P(S|C) \prop P(C|S)P(S)
# multiply the signature matrix (P(C|S) by the exposures matrix P(S)
# choose max for each sample


In [36]:
# load the SVs
# assign SVs to the new signature

In [37]:
# see svig/cataloguePCAWG notebook
pcawg_svs = pd.read_csv('~/projects/rsignatures/data/processed/pcawg.rearrs.annot.csv')

In [38]:
pcawg_svs['label_RFD'] = pcawg_svs['label'] + '_'+ pcawg_svs['rf_dir']

In [39]:
pcawg_svs.head()

,Unnamed: 0,chrom1,start1,end1,chrom2,start2,end2,sv_id,pe_support,strand1,strand2,svclass,svmethod,sample,length,id,is.clustered,bkdist,catalogue.label,label1,label2,label3,label,max.Ref.Sig,sv_id_global,repliseq_bp1,repliseq_bp2,repliseq_avg,repliseq_mp,rf_dir_bp1,rf_dir_bp2,rf_dir,expr_fpkm_bp1,expr_fpkm_bp2,expr_fpkm_avg,origin_density,label_RFD
0,SVMERGE16_51800588-c622-11e3-bf01-24c6515278c0,1,36887569,36887570,1,36892311,36892312,SVMERGE16,49,+,-,deletion,SNOWMAN_BRASS_dRANGER_DELLY,51800588-c622-11e3-bf01-24c6515278c0,4742.0,1,False,4742,non-clustered_del_1-10Kb,non-clustered,_del,_1-10Kb,non-clustered_del_1-10Kb,Ref.Sig.R5,SVMERGE16_51800588-c622-11e3-bf01-24c6515278c0,77.1880,77.0929,77.14045,77.1405,R,R,R_R,3.695367,3.695367,3.695367,0.000057,non-clustered_del_1-10Kb_R_R
1,SVMERGE28_51800588-c622-11e3-bf01-24c6515278c0,1,43007396,43007397,1,43021859,43021860,SVMERGE28,89,+,-,deletion,SNOWMAN_BRASS_DELLY,51800588-c622-11e3-bf01-24c6515278c0,14463.0,2,False,14463,non-clustered_del_10-100Kb,non-clustered,_del,_10-100Kb,non-clustered_del_10-100Kb,Ref.Sig.R5,SVMERGE28_51800588-c622-11e3-bf01-24c6515278c0,66.2956,66.3973,66.34645,66.3437,R,L,R_L,0.399655,0.399655,0.399655,0.000048,non-clustered_del_10-100Kb_R_L
2,SVMERGE1_51800588-c622-11e3-bf01-24c6515278c0,1,59986152,59986153,1,239257903,239257904,SVMERGE1,96,-,-,inversion,SNOWMAN_BRASS_dRANGER_DELLY,51800588-c622-11e3-bf01-24c6515278c0,179271751.0,3,False,179271751,non-clustered_inv_>10Mb,non-clustered,_inv,_>10Mb,non-clustered_inv_>10Mb,Ref.Sig.R5,SVMERGE1_51800588-c622-11e3-bf01-24c6515278c0,30.9046,11.0385,20.97155,63.5684,R,L,R_L,1.494230,0.000000,0.747115,0.000035,non-clustered_inv_>10Mb_R_L
3,SVMERGE2_51800588-c622-11e3-bf01-24c6515278c0,1,59993725,59993726,1,239266604,239266605,SVMERGE2,56,+,+,inversion,SNOWMAN_BRASS_dRANGER_DELLY,51800588-c622-11e3-bf01-24c6515278c0,179272879.0,4,False,179272879,non-clustered_inv_>10Mb,non-clustered,_inv,_>10Mb,non-clustered_inv_>10Mb,Ref.Sig.R5,SVMERGE2_51800588-c622-11e3-bf01-24c6515278c0,30.8760,11.3802,21.12810,63.2716,R,L,R_L,1.494230,0.000000,0.747115,0.000035,non-clustered_inv_>10Mb_R_L
4,SVMERGE27_51800588-c622-11e3-bf01-24c6515278c0,10,6540133,6540134,10,6559698,6559699,SVMERGE27,79,-,+,duplication,SNOWMAN_BRASS_DELLY,51800588-c622-11e3-bf01-24c6515278c0,19565.0,5,False,19565,non-clustered_10-100Kb,non-clustered,_tds,_10-100Kb,non-clustered_tds_10-100Kb,Ref.Sig.R3,SVMERGE27_51800588-c622-11e3-bf01-24c6515278c0,47.5552,45.5983,46.57675,46.5824,L,L,L_L,0.931482,0.931482,0.931482,0.000029,non-clustered_tds_10-100Kb_L_L


In [40]:
pcawg_svs.shape

(246694, 37)

In [41]:
all_sample_channel_assignments.shape

(34432, 3)

In [42]:
all_sample_channel_assignments.head(100)

,label_RFD,sigID,sample
0,non-clustered_del_1-10Kb_L_L,Signature 10,8f558713-f32b-403b-aedf-c79efeb41c67
1,non-clustered_del_1-10Kb_L_R,Signature 11,8f558713-f32b-403b-aedf-c79efeb41c67
2,non-clustered_del_1-10Kb_R_L,Signature 10,8f558713-f32b-403b-aedf-c79efeb41c67
3,non-clustered_del_1-10Kb_R_R,Signature 10,8f558713-f32b-403b-aedf-c79efeb41c67
4,non-clustered_del_10-100Kb_L_L,Signature 11,8f558713-f32b-403b-aedf-c79efeb41c67
...,...,...,...
95,non-clustered_tds_100Kb-1Mb_R_R,Signature 9,1eb62abc-7928-405b-84cc-f091ca5347b2
96,non-clustered_tds_1Mb-10Mb_L_L,Signature 8,1eb62abc-7928-405b-84cc-f091ca5347b2
97,non-clustered_tds_1Mb-10Mb_L_R,Signature 8,1eb62abc-7928-405b-84cc-f091ca5347b2
98,non-clustered_tds_1Mb-10Mb_R_L,Signature 8,1eb62abc-7928-405b-84cc-f091ca5347b2


In [43]:
pcawg_svs[pcawg_svs['sample']=='8f558713-f32b-403b-aedf-c79efeb41c67']['label_RFD'].value_counts()

label_RFD
clustered_trans_L_L                30
clustered_trans_R_L                22
non-clustered_inv_1-10Kb_L_R       21
non-clustered_inv_1-10Kb_L_L       18
clustered_trans_L_R                16
                                   ..
non-clustered_inv_>10Mb_R_R         1
non-clustered_del_100Kb-1Mb_L_R     1
non-clustered_del_10-100Kb_R_L      1
non-clustered_inv_10-100Kb_R_L      1
non-clustered_del_1-10Kb_L_L        1
Name: count, Length: 84, dtype: int64

In [44]:
pcawg_svs_m = pd.merge(pcawg_svs, all_sample_channel_assignments, left_on=['sample', 'label_RFD'], right_on=['sample', 'label_RFD'])

In [45]:
pcawg_svs_m.head()

,Unnamed: 0,chrom1,start1,end1,chrom2,start2,end2,sv_id,pe_support,strand1,strand2,svclass,svmethod,sample,length,id,is.clustered,bkdist,catalogue.label,label1,label2,label3,label,max.Ref.Sig,sv_id_global,repliseq_bp1,repliseq_bp2,repliseq_avg,repliseq_mp,rf_dir_bp1,rf_dir_bp2,rf_dir,expr_fpkm_bp1,expr_fpkm_bp2,expr_fpkm_avg,origin_density,label_RFD,sigID
0,SVMERGE124_7866dfb2-46b3-42b4-905b-12f80593d6bd,1,25946728,25946729,1,28282254,28282255,SVMERGE124,26,+,-,deletion,SNOWMAN_BRASS_dRANGER,7866dfb2-46b3-42b4-905b-12f80593d6bd,2335526.0,1,False,2335526,non-clustered_del_1Mb-10Mb,non-clustered,_del,_1Mb-10Mb,non-clustered_del_1Mb-10Mb,Ref.Sig.R2,SVMERGE124_7866dfb2-46b3-42b4-905b-12f80593d6bd,74.8191,74.8069,74.81300,78.7065,R,R,R_R,1.578989,8.734751,5.156870,0.000089,non-clustered_del_1Mb-10Mb_R_R,Signature 13
1,SVMERGE78_7866dfb2-46b3-42b4-905b-12f80593d6bd,1,73439281,73439282,1,73524194,73524195,SVMERGE78,21,-,+,duplication,SNOWMAN_BRASS_dRANGER,7866dfb2-46b3-42b4-905b-12f80593d6bd,84913.0,2,False,84913,non-clustered_10-100Kb,non-clustered,_tds,_10-100Kb,non-clustered_tds_10-100Kb,Ref.Sig.R9,SVMERGE78_7866dfb2-46b3-42b4-905b-12f80593d6bd,15.2265,14.8904,15.05845,15.1986,R,R,R_R,0.000000,0.000000,0.000000,0.000002,non-clustered_tds_10-100Kb_R_R,Signature 2
2,SVMERGE77_7866dfb2-46b3-42b4-905b-12f80593d6bd,1,85282890,85282891,1,108366593,108366594,SVMERGE77,29,+,-,deletion,SNOWMAN_BRASS_dRANGER_DELLY,7866dfb2-46b3-42b4-905b-12f80593d6bd,23083703.0,3,False,23083703,non-clustered_del_>10Mb,non-clustered,_del,_>10Mb,non-clustered_del_>10Mb,Ref.Sig.R2,SVMERGE77_7866dfb2-46b3-42b4-905b-12f80593d6bd,61.7714,60.1830,60.97720,51.6990,R,R,R_R,1.928794,20.989860,11.459327,0.000007,non-clustered_del_>10Mb_R_R,Signature 12
3,SVMERGE104_7866dfb2-46b3-42b4-905b-12f80593d6bd,1,165753279,165753280,1,165804338,165804339,SVMERGE104,25,+,-,deletion,SNOWMAN_BRASS_dRANGER,7866dfb2-46b3-42b4-905b-12f80593d6bd,51059.0,4,False,51059,non-clustered_del_10-100Kb,non-clustered,_del,_10-100Kb,non-clustered_del_10-100Kb,Ref.Sig.R7,SVMERGE104_7866dfb2-46b3-42b4-905b-12f80593d6bd,52.3661,51.1240,51.74505,51.8748,R,R,R_R,52.013711,12.319274,32.166492,0.000014,non-clustered_del_10-100Kb_R_R,Signature 11
4,SVMERGE172_7866dfb2-46b3-42b4-905b-12f80593d6bd,1,215073016,215073017,1,215076380,215076381,SVMERGE172,30,+,-,deletion,SNOWMAN_BRASS_dRANGER,7866dfb2-46b3-42b4-905b-12f80593d6bd,3364.0,5,False,3364,non-clustered_del_1-10Kb,non-clustered,_del,_1-10Kb,non-clustered_del_1-10Kb,Ref.Sig.R9,SVMERGE172_7866dfb2-46b3-42b4-905b-12f80593d6bd,40.1117,40.0121,40.06190,40.0450,R,R,R_R,0.000000,0.000000,0.000000,0.000014,non-clustered_del_1-10Kb_R_R,Signature 10


In [46]:
pcawg_svs_m['repliseq_mp']
pcawg_svs_m['sigID']

0        Signature 13
1         Signature 2
2        Signature 12
3        Signature 11
4        Signature 10
             ...     
87113    Signature 13
87114    Signature 13
87115     Signature 6
87116     Signature 5
87117    Signature 15
Name: sigID, Length: 87118, dtype: object

In [47]:
# make a boxplot: middpoint replication timing by signature

In [ ]:

# Define desired order for top-to-bottom display
order = ['Signature 4', 'Signature 3', 'Signature 2', 'Signature 1', 'Signature 9', 'Signature 8']

# Add remaining signatures that aren’t in the specified list
other_sigs = [s for s in pcawg_svs_m['sigID'].unique() if s not in order]
full_order = order + sorted(other_sigs)
full_order.reverse()

# To enforce order manually, better use seaborn:
# --- Set PDF backend for vector output ---
plt.rcParams['pdf.fonttype'] = 42   # Ensures fonts are stored as text, not curves
plt.rcParams['ps.fonttype'] = 42    # Same for PostScript
plt.rcParams['svg.fonttype'] = 'none'  # For completeness (if exporting to SVG)

pcawg_svs_m['origin_density_perMb'] = pcawg_svs_m['origin_density'] * (10**6)

plt.figure(figsize=(5, 8))
sns.boxplot(
    data=pcawg_svs_m,
    y='sigID',
    x='origin_density_perMb',
    order=full_order,
    orient='h',
    color='lightblue'
)
plt.xlim(0,0.00011*(10**6))
plt.title("Repli-seq MP by Signature")
plt.xlabel("replication origins per Mb")
plt.ylabel("Signature ID")
plt.gca().invert_yaxis()  # make plot start from top
plt.tight_layout()
plt.savefig("/home/dg204/projects/rsignatures/data/processed/paperPlots/Repliseq_by_Signature_all_perMb.pdf", format="pdf", bbox_inches="tight")

plt.close()
plt.show()

In [49]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define desired order for top-to-bottom display
order = ['Signature 4', 'Signature 3', 'Signature 2', 'Signature 1', 'Signature 9', 'Signature 8']

# Add remaining signatures that aren’t in the specified list
other_sigs = [s for s in pcawg_svs_m['sigID'].unique() if s not in order]
full_order = order
full_order.reverse()

pcawg_svs_m_sel = pcawg_svs_m[pcawg_svs_m['sigID'].isin(order)]

# --- Set PDF backend for vector output ---
plt.rcParams['pdf.fonttype'] = 42   # Ensures fonts are stored as text, not curves
plt.rcParams['ps.fonttype'] = 42    # Same for PostScript
plt.rcParams['svg.fonttype'] = 'none'  # For completeness (if exporting to SVG)

# Create the figure
plt.figure(figsize=(5, 5))
sns.boxplot(
    data=pcawg_svs_m_sel,
    y='sigID',
    x='repliseq_mp',
    order=full_order,
    orient='h',
    color='lightblue'
)

plt.title("Repli-seq MP by Signature")
plt.xlabel("repliseq_mp")
plt.ylabel("Signature ID")
plt.gca().invert_yaxis()  # make plot start from top
plt.tight_layout()

# --- Save as vector PDF with editable text ---
plt.savefig("/home/dg204/projects/rsignatures/data/processed/paperPlots/Repliseq_by_Signature_sel.pdf", format="pdf", bbox_inches="tight")

plt.close()
plt.show()

In [68]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define desired order for top-to-bottom display
order = ['Signature 4', 'Signature 3', 'Signature 2', 'Signature 1', 'Signature 9', 'Signature 8']

# Add remaining signatures that aren’t in the specified list
other_sigs = [s for s in pcawg_svs_m['sigID'].unique() if s not in order]
full_order = order
full_order.reverse()

pcawg_svs_m_sel = pcawg_svs_m[pcawg_svs_m['sigID'].isin(order)]

# --- Set PDF backend for vector output ---
plt.rcParams['pdf.fonttype'] = 42   # Ensures fonts are stored as text, not curves
plt.rcParams['ps.fonttype'] = 42    # Same for PostScript
plt.rcParams['svg.fonttype'] = 'none'  # For completeness (if exporting to SVG)

pcawg_svs_m_sel['origin_density_perMb'] = pcawg_svs_m_sel['origin_density'] * (10**6)

# Create the figure
plt.figure(figsize=(5, 5))
sns.boxplot(
    data=pcawg_svs_m_sel,
    y='sigID',
    x='origin_density_perMb',
    order=full_order,
    orient='h',
    color='lightblue'
)
plt.xlim(0,0.00011*(10**6))
plt.title("Repli-seq MP by Signature")
plt.xlabel("# repl origins per Mb")
plt.ylabel("Signature ID")
plt.gca().invert_yaxis()  # make plot start from top
plt.tight_layout()

plt.savefig("/home/dg204/projects/rsignatures/data/processed/paperPlots/Repli_origin_by_Signature_sel_perMb.pdf", format="pdf", bbox_inches="tight")

plt.close()
plt.show()

/tmp/ipykernel_3692410/3150373772.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pcawg_svs_m_sel['origin_density_perMb'] = pcawg_svs_m_sel['origin_density'] * (10**6)


In [ ]:
from scipy.stats import mannwhitneyu

v_sig1=pcawg_svs_m_sel[pcawg_svs_m_sel['sigID']=='Signature 1']['origin_density']
v_sig1=v_sig1[v_sig1.notna()]
v_sig9=pcawg_svs_m_sel[pcawg_svs_m_sel['sigID']=='Signature 9']['origin_density']
v_sig9=v_sig9[v_sig9.notna()]

stat, p = mannwhitneyu(v_sig1, v_sig9)

In [ ]:
stat


In [ ]:
p

In [ ]:

v_sig8=pcawg_svs_m_sel[pcawg_svs_m_sel['sigID']=='Signature 8']['origin_density']
v_sig8=v_sig8[v_sig8.notna()]

stat, p = mannwhitneyu(v_sig8, v_sig9)

In [ ]:
p